# Stage 0 · GPU notebook

Bounded loader qualification, paired greedy collection, and fixed-context KL/effective writes. All inference uses frozen sources, the recorded runtime and the shared 20-unit budget. Model weights must already exist in the verified local Drive snapshot.


## 1. Storage and checkout
Mount Drive when in Colab; select the existing Stage 0 release checkout. CPU runtime can prepare the plan; one recorded CUDA GPU is required only for the diagnostic run.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

RELEASE_COMMIT = '651d9c3aa9941314374bcf1927860dd2ae2473b6'
REPOSITORY_URL = 'https://github.com/mahadikprasad15/spar-pilot.git'
# Only the dedicated Colab checkout is pinned; local work is never switched.
REPO_DIR = Path('/content/spar-stage0') if Path('/content').is_dir() else Path.cwd()
ARTIFACT_ROOT = Path('/content/drive/MyDrive/SPAR/spar-pilot/artifacts') if Path('/content').is_dir() else REPO_DIR / 'artifacts'
STAGE0_NAME = 'stage0-geometry-v1'
PLAN = ARTIFACT_ROOT / 'plans' / STAGE0_NAME / 'stage0.plan.json'

if Path('/content').is_dir():
    from google.colab import drive
    drive.mount('/content/drive')
if Path('/content').is_dir():
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', REPOSITORY_URL, str(REPO_DIR)], check=True)
    changed = subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO_DIR, text=True)
    if changed.strip():
        raise RuntimeError('The dedicated checkout has local changes. Preserve them before selecting a release.')
    subprocess.run(['git', 'fetch', 'origin', RELEASE_COMMIT], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', '--detach', RELEASE_COMMIT], cwd=REPO_DIR, check=True)
if not (REPO_DIR / 'pilot_eval').is_dir():
    raise RuntimeError('Select the repository checkout before running the workflow.')

def run_stage0(operation, *arguments):
    command = [sys.executable, '-u', '-m', 'pilot_eval', 'stage0-' + operation,
               '--output-root', str(ARTIFACT_ROOT), *map(str, arguments)]
    return subprocess.run(command, cwd=REPO_DIR, check=True)


## 2. Select the pinned dependency environment
Run once in a fresh Colab runtime. Set the install option only when the pinned packages are missing or the next import check fails. Installation streams its output, does not load model weights, and removes the optional vision/audio packages that conflict with this text-only environment. Keep the kernel free of model-library imports while changing these packages.


In [ ]:
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
from pilot_eval.sft_backend import PINS

INSTALL_GPU_DEPENDENCIES = False
if INSTALL_GPU_DEPENDENCIES:
    subprocess.run([sys.executable, '-u', '-m', 'pip', 'install',
                    *[f'{package}=={version}' for package, version in PINS.items()]],
                   cwd=REPO_DIR, check=True)
    subprocess.run([sys.executable, '-u', '-m', 'pip', 'uninstall', '-y',
                    'torchvision', 'torchaudio'], cwd=REPO_DIR, check=True)
print('Required package pins:', PINS)


## 3. Check actual text-model imports
Checks the pinned versions and actual Qwen/PEFT/TRL imports in a fresh child process. It records no scientific qualification and loads no model. CPU preparation may continue without CUDA; the later runtime check requires exactly one visible GPU.


In [ ]:
from pilot_eval.notebook_session import NotebookSession

environment_check = NotebookSession(ARTIFACT_ROOT, REPO_DIR, plan_name=STAGE0_NAME)
ENVIRONMENT = environment_check.check_environment(require_gpu=False)


## 4. Restore source readiness and budget status
Run after setup in a fresh kernel. Unknown prior Colab usage is not treated as zero.


In [ ]:
run_stage0('status', '--config', PLAN)


## 5. Prepare the qualification plan (CPU safe)
This binds the completed Ticket 03 adapter catalogue to the original ordered Pilot 4 150-question evaluation config. It does not load a model or use the GPU.

In [ ]:
EDIT_NAME = STAGE0_NAME + '-principal-edits-v1'
EDIT_PLAN = ARTIFACT_ROOT / 'plans' / EDIT_NAME / 'stage0.edits.json'
PILOT4_EVALUATION_CONFIG = ARTIFACT_ROOT / 'reports' / 'pilot4-results-so-far-2026-10-13' / 'inputs' / 'evaluation-config.json'
QUALIFICATION_NAME = STAGE0_NAME + '-loader-qualification-v1'
QUALIFICATION_PLAN = ARTIFACT_ROOT / 'plans' / QUALIFICATION_NAME / 'stage0.qualification.json'
assert EDIT_PLAN.is_file(), 'Complete CPU notebook sections 10–12 and restore the edit catalogue first.'
assert PILOT4_EVALUATION_CONFIG.is_file(), 'Restore the original Pilot 4 evaluation-config.json under artifacts/reports/.../inputs/.'
run_stage0('qualification-prepare', '--edits', EDIT_PLAN, '--cohort', PILOT4_EVALUATION_CONFIG, '--name', QUALIFICATION_NAME)


## 6. Check this GPU session and Drive readback
This cell checks the pinned runtime, visible single GPU, stable session ID, budget ledger, and a small write/read/delete on Drive. It does not load the model or spend inference time.

In [ ]:
def run_stage0_json(operation, *arguments):
    command = [sys.executable, '-u', '-m', 'pilot_eval', 'stage0-' + operation, '--output-root', str(ARTIFACT_ROOT), *map(str, arguments)]
    completed = subprocess.run(command, cwd=REPO_DIR, check=True, capture_output=True, text=True)
    value = json.loads(completed.stdout)
    print(json.dumps(value, indent=2))
    return value

PREFLIGHT = run_stage0_json('qualification-runtime', '--config', QUALIFICATION_PLAN)
QUALIFICATION_SESSION_ID = PREFLIGHT['runtime']['session_id']
assert PREFLIGHT['storage']['passed']
print('Prior cumulative usage:', PREFLIGHT['budget'])
assert PREFLIGHT['budget']['spent_units'] < 20, 'The cumulative Stage 0 ceiling is reached.'
assert not PREFLIGHT['active_reservations'], 'Reconcile the earlier reservation before starting another GPU job.'


## 7. Record the current cumulative Stage 0 usage (CPU safe)
Read the current Colab compute-unit total and enter it below. This is a cumulative Stage 0 reading across sessions, not the account balance. If you cannot establish prior usage, stop here and recover that number before reserving work.

In [ ]:
from datetime import datetime, timezone
import uuid

OBSERVED_STAGE0_UNITS = None  # Enter the actual cumulative Stage 0 compute-unit reading.
OBSERVATION_EVENT_ID = 'ticket04-start-' + uuid.uuid4().hex
if OBSERVED_STAGE0_UNITS is None:
    raise ValueError('Enter the observed cumulative Stage 0 usage; unknown is not zero.')
observation = {
    'event_id': OBSERVATION_EVENT_ID,
    'session_id': QUALIFICATION_SESSION_ID,
    'total_units': float(OBSERVED_STAGE0_UNITS),
    'observed_at': datetime.now(timezone.utc).isoformat(),
    'note': 'Operator-recorded cumulative Stage 0 Colab compute units before Ticket 04.'
}
OBSERVATION_PATH = ARTIFACT_ROOT / 'notebook' / STAGE0_NAME / 'ticket04-usage-observation.json'
OBSERVATION_PATH.parent.mkdir(parents=True, exist_ok=True)
OBSERVATION_PATH.write_text(json.dumps(observation, indent=2) + '\n')
run_stage0('budget-record', '--config', PLAN, '--observation', OBSERVATION_PATH)
PREFLIGHT = run_stage0_json('qualification-runtime', '--config', QUALIFICATION_PLAN)
assert PREFLIGHT['budget']['state'] == 'observed' and PREFLIGHT['budget']['spent_units'] < 20


## 8. Approve and run the bounded diagnostic
Review the session and remaining budget from section 7. Set a conservative Colab unit rate and a wall-time cap for this diagnostic. The reservation is calculated before model load; interrupted work stays reserved until a newer cumulative usage reading is recorded.

Each of the 32 units compares an adapter reference with the mixed merged loader, the intentional double-application negative control, and greedy outputs. The run saves each completed unit and resumes only missing units.

In [ ]:
DIAGNOSTIC_REVIEWER = 'Prasad'
DIAGNOSTIC_NOTES = 'Bounded loader qualification; reviewed current Stage 0 usage and the 32-unit scope.'
DIAGNOSTIC_SOURCE_REVIEWED = False  # Set True only after confirming the exact edits and cohort paths above.
DIAGNOSTIC_MAX_UNITS_PER_HOUR = None  # Enter a conservative rate based on the current Colab runtime.
DIAGNOSTIC_MAX_SECONDS = None  # Enter a hard wall-time cap for this diagnostic attempt.
if not DIAGNOSTIC_SOURCE_REVIEWED or DIAGNOSTIC_MAX_UNITS_PER_HOUR is None or DIAGNOSTIC_MAX_SECONDS is None:
    raise ValueError('Review the sources and enter conservative unit-rate and wall-time bounds before reserving GPU work.')
DIAGNOSTIC_APPROVAL = {
    'reviewer': DIAGNOSTIC_REVIEWER, 'notes': DIAGNOSTIC_NOTES,
    'session_id': QUALIFICATION_SESSION_ID,
    'max_units_per_hour': float(DIAGNOSTIC_MAX_UNITS_PER_HOUR),
    'max_seconds': float(DIAGNOSTIC_MAX_SECONDS),
    'source_reviewed': DIAGNOSTIC_SOURCE_REVIEWED
}
DIAGNOSTIC_APPROVAL_PATH = ARTIFACT_ROOT / 'notebook' / STAGE0_NAME / 'ticket04-diagnostic-approval.json'
DIAGNOSTIC_APPROVAL_PATH.write_text(json.dumps(DIAGNOSTIC_APPROVAL, indent=2) + '\n')
run_stage0('qualification-run', '--config', QUALIFICATION_PLAN, '--approval', DIAGNOSTIC_APPROVAL_PATH)


## 9. Verify and inspect calibration evidence
Verification is CPU-only. It checks all 32 saved units, token/mask identities, runtime binding, negative-control sensitivity and the hard stop. It does not itself approve the proposed 3E numerical allowance.

In [ ]:
run_stage0('qualification-verify', '--config', QUALIFICATION_PLAN)
DIAGNOSTIC_RESULT_PATH = ARTIFACT_ROOT / 'runs' / 'stage0' / QUALIFICATION_NAME / 'qualification' / 'results' / 'diagnostic.json'
DIAGNOSTIC_RESULT = json.loads(DIAGNOSTIC_RESULT_PATH.read_text())
print(json.dumps({key: DIAGNOSTIC_RESULT[key] for key in ['max_absolute_error', 'hard_stop', 'negative_control', 'greedy_acceptable', 'eligible_for_review', 'proposed_absolute_gate', 'production_estimate']}, indent=2))


## 9A. Reconcile diagnostic compute usage before resuming or freezing
After the diagnostic completes or stops, record the updated cumulative Stage 0 usage. This releases the diagnostic reservation while preserving the run's measured delta. You can record the reading from a later session after an interruption; the ledger records both run-session and observation-session IDs.

If the diagnostic is incomplete and the same Colab runtime is still connected, reconcile here before rerunning section 8. Sealed units resume only within that recorded runtime/session. If Colab restarted or the GPU/runtime changed, preserve this attempt and prepare a new named qualification; do not mix partial results across runtimes.

In [ ]:
CUMULATIVE_STAGE0_UNITS_AFTER_DIAGNOSTIC = None  # Enter the actual current cumulative Stage 0 unit reading.
if CUMULATIVE_STAGE0_UNITS_AFTER_DIAGNOSTIC is None:
    raise ValueError('Enter the observed cumulative usage before releasing the reservation.')
STATUS_PATH = ARTIFACT_ROOT / 'runs' / 'stage0' / QUALIFICATION_NAME / 'qualification' / 'meta' / 'status.json'
DIAGNOSTIC_RESULT_PATH = ARTIFACT_ROOT / 'runs' / 'stage0' / QUALIFICATION_NAME / 'qualification' / 'results' / 'diagnostic.json'
ATTEMPT_STATUS = json.loads(STATUS_PATH.read_text())
DIAGNOSTIC_RESULT = json.loads(DIAGNOSTIC_RESULT_PATH.read_text()) if DIAGNOSTIC_RESULT_PATH.exists() else {}
RESERVATION_ID = DIAGNOSTIC_RESULT.get('reservation_id') or ATTEMPT_STATUS.get('reservation_id')
assert RESERVATION_ID, 'No diagnostic reservation found to reconcile.'
OBSERVATION_PATH = ARTIFACT_ROOT / 'notebook' / STAGE0_NAME / 'ticket04-usage-observation.json'
observation = {
    'event_id': 'ticket04-after-diagnostic-' + uuid.uuid4().hex,
    'session_id': __import__('platform').node() + ':' + __import__('pathlib').Path('/proc/sys/kernel/random/boot_id').read_text().strip(),
    'total_units': float(CUMULATIVE_STAGE0_UNITS_AFTER_DIAGNOSTIC),
    'observed_at': datetime.now(timezone.utc).isoformat(),
    'note': 'Operator-recorded cumulative Stage 0 compute units after Ticket 04 diagnostic attempt.'
}
OBSERVATION_PATH.write_text(json.dumps(observation, indent=2) + '\n')
run_stage0('budget-record', '--config', PLAN, '--observation', OBSERVATION_PATH)
run_stage0('budget-settle', '--reservation-id', RESERVATION_ID)
PREFLIGHT = run_stage0_json('qualification-runtime', '--config', QUALIFICATION_PLAN)
assert PREFLIGHT['budget']['spent_units'] < 20, 'No unapproved work remains under the Stage 0 ceiling.'


## 10. Save the measured numerical-gate review
Only approve after reviewing the saved maximum error, every answer-position maximum, the worst coordinate, negative-control ratios and any common-prefix near-tie exceptions. Enter the exact evidence-derived 3E allowance shown below. A diagnostic above 1e-3 or an insensitive negative control cannot be approved.

In [ ]:
NUMERICAL_REVIEWED = False  # Set True after inspecting the complete diagnostic JSON and saved unit records.
NUMERICAL_REVIEWER = 'Prasad'
NUMERICAL_REVIEW_NOTES = ''
if not NUMERICAL_REVIEWED or not NUMERICAL_REVIEW_NOTES.strip() or not DIAGNOSTIC_RESULT['eligible_for_review']:
    raise ValueError('Review eligible diagnostic evidence and enter notes before approving the numerical allowance.')
DIAGNOSTIC_COMPLETE = ARTIFACT_ROOT / 'runs' / 'stage0' / QUALIFICATION_NAME / 'qualification' / 'complete.json'
NUMERICAL_REVIEW_PATH = ARTIFACT_ROOT / 'notebook' / STAGE0_NAME / 'ticket04-numerical-review.json'
NUMERICAL_REVIEW = {
    'reviewer': NUMERICAL_REVIEWER, 'notes': NUMERICAL_REVIEW_NOTES,
    'diagnostic_sha256': __import__('hashlib').sha256(DIAGNOSTIC_COMPLETE.read_bytes()).hexdigest(),
    'absolute_gate': DIAGNOSTIC_RESULT['proposed_absolute_gate']
}
NUMERICAL_REVIEW_PATH.write_text(json.dumps(NUMERICAL_REVIEW, indent=2) + '\n')
run_stage0('qualification-approve', '--config', QUALIFICATION_PLAN, '--review', NUMERICAL_REVIEW_PATH)


## 11. Approve the measured production reservation and freeze
Enter a conservative runtime unit rate and wall-time allowance that covers the measured production forecast. The frozen plan covers the ten-variant, 150-question paired greedy evaluation; later KL/effective-write jobs need their own measured reservations under the same 20-unit ceiling.

In [ ]:
PRODUCTION_REVIEWED = False  # Set True after reviewing the full measured forecast.
PRODUCTION_REVIEWER = 'Prasad'
PRODUCTION_REVIEW_NOTES = ''
PRODUCTION_MAX_UNITS_PER_HOUR = None  # Enter a conservative rate from Colab's current compute-unit display.
PRODUCTION_MAX_SECONDS = None  # Must cover conservative_total_seconds below.
if (not PRODUCTION_REVIEWED or not PRODUCTION_REVIEW_NOTES.strip()
        or PRODUCTION_MAX_UNITS_PER_HOUR is None or PRODUCTION_MAX_SECONDS is None):
    raise ValueError('Review the forecast and enter explicit unit/time bounds before freezing production work.')
assert PRODUCTION_MAX_SECONDS >= DIAGNOSTIC_RESULT['production_estimate']['conservative_total_seconds']
PRODUCTION_REVIEW_PATH = ARTIFACT_ROOT / 'notebook' / STAGE0_NAME / 'ticket04-production-budget-review.json'
PRODUCTION_REVIEW = {
    'reviewer': PRODUCTION_REVIEWER, 'notes': PRODUCTION_REVIEW_NOTES,
    'diagnostic_sha256': __import__('hashlib').sha256(DIAGNOSTIC_COMPLETE.read_bytes()).hexdigest(),
    'max_units_per_hour': float(PRODUCTION_MAX_UNITS_PER_HOUR),
    'max_seconds': float(PRODUCTION_MAX_SECONDS),
    'session_id': DIAGNOSTIC_RESULT['runtime']['session_id'],
    'source_reviewed': True
}
PRODUCTION_REVIEW_PATH.write_text(json.dumps(PRODUCTION_REVIEW, indent=2) + '\n')
run_stage0('qualification-freeze', '--config', QUALIFICATION_PLAN, '--review', PRODUCTION_REVIEW_PATH)


## 12. Restore and prepare paired greedy collection (CPU safe)
Consumes the frozen Ticket 04 execution. Inference stays at its qualified batch size of 1; eight sequential calls form one durable unit. The full catalogue is 10 variants × 150 questions. This cell displays the frozen work count and does not load a model. After a kernel restart, run section 1 and this section to restore the saved paths.


In [ ]:
QUALIFICATION_NAME = STAGE0_NAME + '-loader-qualification-v1'
EXECUTION_PLAN = ARTIFACT_ROOT / 'plans' / QUALIFICATION_NAME / 'stage0.execution.json'
EVALUATION_PLAN = EXECUTION_PLAN.parent / 'stage0.evaluation.json'
INFERENCE_BATCH_SIZE = 1
BATCHES_PER_DURABLE_UNIT = 8
assert EXECUTION_PLAN.is_file(), 'Complete the numerical and production-budget reviews and freeze in sections 10–11 first.'
run_stage0('evaluation-prepare', '--execution', EXECUTION_PLAN,
           '--batch-size', INFERENCE_BATCH_SIZE, '--batches-per-unit', BATCHES_PER_DURABLE_UNIT)
EVALUATION_CONFIG = json.loads(EVALUATION_PLAN.read_text())
EVALUATION_DIR = ARTIFACT_ROOT / EVALUATION_CONFIG['run_path']
print('Variants:', EVALUATION_CONFIG['variant_order'])
print('Questions:', len(EVALUATION_CONFIG['item_ids']))
print('Model generation calls:', sum(len(unit['generation_batches']) for unit in EVALUATION_CONFIG['units']))
print('Durable units:', len(EVALUATION_CONFIG['units']))
print('Raw results:', EVALUATION_DIR / 'results')


## 12A. Freeze the primary comparison plan before collection (CPU safe)
Freezes the full comparison family, hypotheses, bootstrap procedure and −5 pp preservation rule before new results. Preparing the drift plan loads no model. If collection already started, the analysis is marked exploratory. The statistical NumPy runtime is recorded when the CPU analysis actually runs, independently of this GPU preparation environment.


In [ ]:
QUALIFICATION_NAME = STAGE0_NAME + '-loader-qualification-v1'
EXECUTION_PLAN = ARTIFACT_ROOT / 'plans' / QUALIFICATION_NAME / 'stage0.execution.json'
EVALUATION_PLAN = EXECUTION_PLAN.parent / 'stage0.evaluation.json'
DRIFT_NAME = STAGE0_NAME + '-fixed-context-v1'
DRIFT_PLAN = ARTIFACT_ROOT / 'plans' / DRIFT_NAME / 'stage0.drift.json'
FINDINGS_NAME = STAGE0_NAME + '-primary-findings-v1'
FINDINGS_PLAN = ARTIFACT_ROOT / 'plans' / FINDINGS_NAME / 'stage0.findings.json'
FIXED_INPUTS = ARTIFACT_ROOT / 'plans' / 'pilot3-inputs-v1' / 'activation.prepared.json'
run_stage0('drift-prepare', '--execution', EXECUTION_PLAN, '--inputs', FIXED_INPUTS,
           '--name', DRIFT_NAME, '--batch-size', 1, '--batches-per-unit', 8,
           '--context-chunk', 32, '--workspace-bytes', 512 * 1024 * 1024)
run_stage0('findings-prepare', '--evaluation', EVALUATION_PLAN, '--drift', DRIFT_PLAN,
           '--name', FINDINGS_NAME)
registration = json.loads(FINDINGS_PLAN.read_text())
print('Registration:', registration['registration'])
print('Comparisons:', [row['id'] for row in registration['contrasts']])


## 13. Collect or resume paired greedy responses (GPU)
Loads one local FP32 model and the ten verified adapters in the recorded qualified runtime. It re-evaluates the SFT and GRPO references alongside every edit, then seals responses, token IDs, EOS/cap metadata, strict/flexible-v3 scores and provenance. Progress distinguishes generated responses pending a seal from durable completed units.

The initial run consumes section 11's active reservation. A normal same-session resume can use its remaining recorded wall time. After reconciling a stopped or exhausted reservation in section 14, explicitly approve a new bounded attempt using the settings below. A different runtime/session is rejected before loading the model.


In [ ]:
RENEW_RESERVATION = False
RESUME_REVIEWER = 'Prasad'
RESUME_REVIEW_NOTES = ''
RESUME_MAX_UNITS_PER_HOUR = None
RESUME_MAX_SECONDS = None

arguments = ['--config', EVALUATION_PLAN]
if RENEW_RESERVATION:
    if (not RESUME_REVIEW_NOTES.strip() or RESUME_MAX_UNITS_PER_HOUR is None
            or RESUME_MAX_SECONDS is None):
        raise ValueError('Enter reviewed notes and conservative rate/time limits after reconciling the earlier reservation.')
    review = {
        'reviewer': RESUME_REVIEWER, 'notes': RESUME_REVIEW_NOTES,
        'session_id': EVALUATION_CONFIG['runtime']['session_id'], 'source_reviewed': True,
        'max_units_per_hour': float(RESUME_MAX_UNITS_PER_HOUR),
        'max_seconds': float(RESUME_MAX_SECONDS),
    }
    RESUME_REVIEW_PATH = ARTIFACT_ROOT / 'notebook' / STAGE0_NAME / 'paired-resume-budget-review.json'
    RESUME_REVIEW_PATH.parent.mkdir(parents=True, exist_ok=True)
    RESUME_REVIEW_PATH.write_text(json.dumps(review, indent=2) + '\n')
    arguments += ['--approval', RESUME_REVIEW_PATH]
run_stage0('evaluation-run', *arguments)


## 14. Reconcile collection usage (CPU safe)
After collection completes or stops, enter the actual cumulative Stage 0 compute-unit usage to reconcile its reservation. This does not change responses or reset the total budget. After a lost process, inspect `evaluation-status` and the saved writer token before using the documented explicit writer-recovery command.


In [ ]:
from datetime import datetime, timezone
import platform
import uuid

CUMULATIVE_STAGE0_UNITS_AFTER_COLLECTION = None
if CUMULATIVE_STAGE0_UNITS_AFTER_COLLECTION is None:
    raise ValueError('Enter the actual cumulative Stage 0 usage before settling this reservation.')
allocation = json.loads((EVALUATION_DIR / 'meta' / 'active-reservation.json').read_text())
observation = {
    'event_id': 'paired-after-' + uuid.uuid4().hex,
    'session_id': platform.node() + ':' + Path('/proc/sys/kernel/random/boot_id').read_text().strip(),
    'total_units': float(CUMULATIVE_STAGE0_UNITS_AFTER_COLLECTION),
    'observed_at': datetime.now(timezone.utc).isoformat(),
    'note': 'Operator-observed cumulative Stage 0 consumption after paired collection attempt.'
}
OBSERVATION_PATH = ARTIFACT_ROOT / 'notebook' / STAGE0_NAME / 'paired-usage-observation.json'
OBSERVATION_PATH.parent.mkdir(parents=True, exist_ok=True)
OBSERVATION_PATH.write_text(json.dumps(observation, indent=2) + '\n')
run_stage0('budget-record', '--config', PLAN, '--observation', OBSERVATION_PATH)
run_stage0('budget-settle', '--reservation-id', allocation['reservation_id'])
run_stage0('evaluation-status', '--config', EVALUATION_PLAN)


## 15. Verify and inspect the raw result (CPU)
Reconstructs scores and summary counts from every sealed batch and checks the aggregate response JSONL. Run this on CPU in the second notebook after GPU collection and usage reconciliation. Paired intervals, human-answer overlays and accuracy-preservation claims belong to Ticket 07.


In [ ]:
run_stage0('evaluation-verify', '--config', EVALUATION_PLAN)
from IPython.display import Markdown, display
summary = json.loads((EVALUATION_DIR / 'results' / 'results.json').read_text())
print('Verified responses:', summary['response_count'])
print('Execution kind:', summary['execution_kind'])
display(Markdown((EVALUATION_DIR / 'results' / 'results.md').read_text()))
print('Raw JSONL:', EVALUATION_DIR / 'results' / 'responses.jsonl')
print('Structured summary:', EVALUATION_DIR / 'results' / 'results.json')


## 16. Prepare fixed-context KL and writes (CPU safe)
Uses the original 150 GSM8K gold and 150 FineWeb sequences and all ten primary variants. Three bounded instrument controls use four documents each. Inference remains singleton; saving eight documents together does not change the model batch. Section 15 can be deferred to the CPU notebook. Reconcile the greedy reservation before starting the next GPU job.


In [ ]:
QUALIFICATION_NAME = STAGE0_NAME + '-loader-qualification-v1'
EXECUTION_PLAN = ARTIFACT_ROOT / 'plans' / QUALIFICATION_NAME / 'stage0.execution.json'
FIXED_INPUTS = ARTIFACT_ROOT / 'plans' / 'pilot3-inputs-v1' / 'activation.prepared.json'
DRIFT_NAME = STAGE0_NAME + '-fixed-context-v1'
DRIFT_PLAN = ARTIFACT_ROOT / 'plans' / DRIFT_NAME / 'stage0.drift.json'
DRIFT_CONTEXT_CHUNK = 32
DRIFT_WORKSPACE_BYTES = 512 * 1024 * 1024
run_stage0('drift-prepare', '--execution', EXECUTION_PLAN, '--inputs', FIXED_INPUTS,
           '--name', DRIFT_NAME, '--batch-size', 1, '--batches-per-unit', 8,
           '--context-chunk', DRIFT_CONTEXT_CHUNK, '--workspace-bytes', DRIFT_WORKSPACE_BYTES)
DRIFT_CONFIG = json.loads(DRIFT_PLAN.read_text())
DRIFT_DIR = ARTIFACT_ROOT / DRIFT_CONFIG['run_path']
print('Main variant/document combinations:', 10 * len(DRIFT_CONFIG['item_ids']))
print('Durable units:', len(DRIFT_CONFIG['units']))
print('Bounded controls:', DRIFT_CONFIG['controls'])
print('Saved results:', DRIFT_DIR / 'results')


## 17. Bound, measure or resume fixed contexts (GPU)
Approve a separate allocation after recording actual cumulative usage and settling the previous reservation. The initial limit of four units collects three controls and the first SFT group; inspect its timing before continuing. Set the unit limit to `None` to finish under the remaining approved time. A new allocation requires usage reconciliation; an existing active allocation resumes without a new approval. Full-vocabulary probabilities are reduced in bounded chunks and discarded.


In [ ]:
APPROVE_NEW_DRIFT_BUDGET = False
DRIFT_MAX_UNITS_PER_HOUR = None
DRIFT_MAX_SECONDS = None
DRIFT_REVIEW_NOTES = ''
DRIFT_MAX_NEW_UNITS = 4
DRIFT_APPROVAL = ARTIFACT_ROOT / 'notebook' / STAGE0_NAME / 'drift-budget-review.json'
arguments = ['--config', DRIFT_PLAN]
if APPROVE_NEW_DRIFT_BUDGET:
    if DRIFT_MAX_UNITS_PER_HOUR is None or DRIFT_MAX_SECONDS is None or not DRIFT_REVIEW_NOTES.strip():
        raise ValueError('Enter a conservative current unit rate, bounded seconds and review notes.')
    import platform
    approval = {
        'reviewer': 'Prasad', 'notes': DRIFT_REVIEW_NOTES,
        'session_id': platform.node() + ':' + Path('/proc/sys/kernel/random/boot_id').read_text().strip(),
        'max_units_per_hour': float(DRIFT_MAX_UNITS_PER_HOUR),
        'max_seconds': float(DRIFT_MAX_SECONDS), 'source_reviewed': True
    }
    DRIFT_APPROVAL.parent.mkdir(parents=True, exist_ok=True)
    DRIFT_APPROVAL.write_text(json.dumps(approval, indent=2) + '\n')
    arguments += ['--approval', DRIFT_APPROVAL]
if DRIFT_MAX_NEW_UNITS is not None:
    arguments += ['--max-new-units', DRIFT_MAX_NEW_UNITS]
run_stage0('drift-run', *arguments)
run_stage0('drift-status', '--config', DRIFT_PLAN)


## 18. Reconcile fixed-context usage (CPU safe)
Record the actual cumulative Stage 0 usage after completion or a stopped attempt. Settling the allocation does not delete sealed units. If more work is needed, a later bounded allocation resumes those units in the same qualified runtime.


In [ ]:
from datetime import datetime, timezone
import platform
import uuid

CUMULATIVE_STAGE0_UNITS_AFTER_DRIFT = None
if CUMULATIVE_STAGE0_UNITS_AFTER_DRIFT is None:
    raise ValueError('Enter actual cumulative Stage 0 compute-unit usage.')
allocation = json.loads((DRIFT_DIR / 'meta' / 'active-reservation.json').read_text())
observation = {
    'event_id': 'drift-after-' + uuid.uuid4().hex,
    'session_id': platform.node() + ':' + Path('/proc/sys/kernel/random/boot_id').read_text().strip(),
    'total_units': float(CUMULATIVE_STAGE0_UNITS_AFTER_DRIFT),
    'observed_at': datetime.now(timezone.utc).isoformat(),
    'note': 'Operator-observed cumulative Stage 0 usage after fixed-context attempt.'
}
OBSERVATION_PATH = ARTIFACT_ROOT / 'notebook' / STAGE0_NAME / 'drift-usage-observation.json'
OBSERVATION_PATH.parent.mkdir(parents=True, exist_ok=True)
OBSERVATION_PATH.write_text(json.dumps(observation, indent=2) + '\n')
run_stage0('budget-record', '--config', PLAN, '--observation', OBSERVATION_PATH)
run_stage0('budget-settle', '--reservation-id', allocation['reservation_id'])
run_stage0('drift-status', '--config', DRIFT_PLAN)


## 19. Verify fixed-context results (CPU)
Reconstruct the document and corpus/view summaries from sealed scalar KL arrays and module write sums. No model is loaded. Equal-document means and token-weighted means remain separate; uncertainty and joined behavioral claims belong to Ticket 07.


In [ ]:
run_stage0('drift-verify', '--config', DRIFT_PLAN)
result = json.loads((DRIFT_DIR / 'results' / 'results.json').read_text())
print('Measurement complete:', result['measurement_complete'])
print('Execution kind:', result['execution_kind'])
print('Corpus/view cells:', len(result['cells']))
print('Per-document raw sums:', DRIFT_DIR / 'results' / 'per-document.jsonl')
print('Per-position KL and mappings:', DRIFT_DIR / 'units')


## CPU handoff
After GPU sections 13–14 and 17–18 finish collection and reconcile usage, release the GPU. CPU notebook sections 17–22 produce the primary report, optional human-label overlay, and raw data/figure exports. Supporting experiments and the conditional intruder extension do not block these primary findings.
